Key Deliverables 
- EntityExtractor class with regex patterns for numeric entities ✅
- Amenity detection using Week 1 taxonomy ✅
- Labeled dataset: 200-300 remarks with entity spans 
- Evaluation script calculating precision/recall/F1 
- Target: 85%+ F1 score on test set 
- Error analysis identifying top failure patterns 

Objective: Build a program that reads a real estate listing description/remarks and pull out important information into a structured format

In [515]:
import re 
from collections import Counter
import json
import pandas as pd
import sys
sys.path.append("../scripts")

In [516]:
from w2_text_cleaning import TextCleaner

In [517]:
with open("../data/processed/taxonomy.json", "r") as f:
    taxonomy = json.load(f)
print("Total terms:", len(taxonomy["terms"]))
print(taxonomy["categories"])
print()
term_counts = Counter(term["category"] for term in taxonomy["terms"])

print("Term Counts per Category")
for category in taxonomy["categories"]:
    count = term_counts.get(category, 0)
    print(f"{category}: {count} terms")

Total terms: 200
['kitchen', 'flooring', 'outdoor', 'parking', 'housing_layout', 'housing_features', 'location', 'community_amenities']

Term Counts per Category
kitchen: 22 terms
flooring: 13 terms
outdoor: 31 terms
parking: 20 terms
housing_layout: 43 terms
housing_features: 41 terms
location: 28 terms
community_amenities: 2 terms


In [518]:
print(taxonomy['terms'])

[{'id': 1, 'term': 'primary suite', 'category': 'housing_layout', 'frequency': 336}, {'id': 2, 'term': 'natural light', 'category': 'housing_features', 'frequency': 294}, {'id': 3, 'term': 'living room', 'category': 'housing_layout', 'frequency': 283}, {'id': 4, 'term': 'living space', 'category': 'outdoor', 'frequency': 263}, {'id': 5, 'term': 'floor plan', 'category': 'flooring', 'frequency': 261}, {'id': 6, 'term': 'stainless steel', 'category': 'kitchen', 'frequency': 193}, {'id': 7, 'term': 'everyday living', 'category': 'parking', 'frequency': 186}, {'id': 8, 'term': 'stainless steel appliances', 'category': 'kitchen', 'frequency': 172}, {'id': 9, 'term': 'shopping dining', 'category': 'location', 'frequency': 171}, {'id': 10, 'term': 'located near', 'category': 'location', 'frequency': 159}, {'id': 11, 'term': 'conveniently located', 'category': 'location', 'frequency': 158}, {'id': 12, 'term': 'home office', 'category': 'housing_layout', 'frequency': 143}, {'id': 13, 'term': 'p

In [519]:
class EntityExtractor: 
    def __init__(self):
        with open("../data/processed/taxonomy.json", "r") as f:
            taxonomy = json.load(f)
        self.taxonomy = taxonomy['terms']
    def extract_bedrooms(self, text): 
        patterns = [ 
        r'(\d+)\s*(?:bed|beds|br|bd|bdrm|bdrms|bedroom|bedrooms)\b'
        ] 
        for pattern in patterns: 
            match = re.search(pattern, text, re.I) 
            if match: 
                return int(match.group(1)) 
        return None 
    def extract_price(self, text): 
        # Assumes cleaned text from Week 2 
        match = re.search(r'\$?(\d{5,})', text) 
        return int(match.group(1)) if match else None 
    def extract_bathrooms(self, text):
        patterns =  [
            r'(\d+\.?\d*)\s*(?:bath|baths|bathroom|bathrooms|ba)\b'
        ]
        for pattern in patterns: 
            match = re.search(pattern, text, re.I) 
            if match: 
                return int(match.group(1)) 
        return None 
    def extract_sqft(self, text): 
        patterns = [
            r"\b(sq\.?\s*ft\.?|sqft|sf|square feet)\b"
        ]
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                cleaned_value = match.group(1).replace(",", "") # remove commas
                return int(match.group(1)) 
        return None 
    def extract_amenities(self, text):
        amenities = []
        text = text.lower()
        for item in self.taxonomy:
            term = item["term"].lower()
            if term in text:
                amenities.append({
                    "term": item["term"],
                    "category": item["category"]
                })
        return amenities
    def extract_all(self, text): 
        return { 
        'bedrooms': self.extract_bedrooms(text), 
        'bathrooms': self.extract_bathrooms(text), 
        'price': self.extract_price(text), 
        'sqft': self.extract_sqft(text), 
        'amenities': self.extract_amenities(text) 
        } 


Test cases

In [520]:
extractor = EntityExtractor()

In [521]:
df_cleaned = pd.read_csv("../data/processed/cleaned_listing_sample.csv")
df_cleaned.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,This unique property offers two homes on one l...,This unique property offers two homes on one l...
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,Beautiful Two-Story Home in Victorville – Spac...,Beautiful Two-Story Home in Victorville - Spac...
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,Presenting this exquisite second-floor corner ...,Presenting this exquisite second-floor corner ...
3,1170333192,1872 Seascape Boulevard,Aptos,3.0,3.0,1195000,Thoughtfully renovated coastal retreat tucked ...,Thoughtfully renovated coastal retreat tucked ...
4,1152463169,2085 Westhampton Drive,Arroyo Grande,3.0,2.0,1050000,Welcome to 2085 Westhampton Drive in Arroyo Gr...,Welcome to 2085 Westhampton Drive in Arroyo Gr...


In [522]:
df_cleaned[['remarks']].head(1)

,remarks
0,This unique property offers two homes on one l...


Checking extracting amenities

In [523]:
remark = """
Beautiful 4 bedroom home with granite countertops,
luxury vinyl plank flooring, pool, and RV parking.
"""

In [524]:
print(extractor.extract_amenities(remark))

[{'term': 'granite countertops', 'category': 'kitchen'}, {'term': 'plank flooring', 'category': 'flooring'}, {'term': 'vinyl plank flooring', 'category': 'flooring'}, {'term': 'rv parking', 'category': 'parking'}]


Extracting all

In [525]:
df_cleaned.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price',
       'remarks', 'cleaned_remarks'],
      dtype='str')

In [526]:
sample_remark = df_cleaned['cleaned_remarks'].iloc[0]

In [527]:
sample_remark

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [528]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


With this sample, only the first mentions of the bedrooms and bathrooms were extracted. Looking at the remark, they expanded on what the rear unit consists of. 

In [529]:
example_remark = '''
There are 2.5 beds and 3 bathrooms.
'''

In [530]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


#### Labeled dataset: 200-300 remarks with entity spans 

In [531]:
labeled_sample = df_cleaned.sample(n=250, random_state=422)

In [532]:
#labeled_sample.to_csv('../data/labeled_sample.csv', index=False)
print(len(labeled_sample))

250


In [533]:
labeled_sample_cleaned = labeled_sample.drop(columns=['L_Address', 'remarks', 'L_City', 'L_ListingID'])

In [534]:
labeled_sample_cleaned.head()

,beds,baths,price,cleaned_remarks
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits..."
255,2.0,3.0,389000,Multi-level townhouse looking for a little lov...
258,4.0,4.0,2625000,Experience refined Central Coast living in thi...
301,3.0,4.0,3325000,Wine country estate set across 52 acres in the...
233,3.0,2.0,650000,Welcome to this beautifully upgraded Connect f...


sqft (ground truth)

In [535]:
labeled_sample_cleaned['sqft'] = None

In [536]:
sqft_rows = labeled_sample_cleaned[
    labeled_sample_cleaned["cleaned_remarks"].str.contains(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        case=False,
        na=False,
        regex=True
    )
]
len(sqft_rows)

87

In [537]:
def sqft_context(text, window=35):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)

    return text[start:end]

In [538]:
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)

In [539]:
labeled_sample_cleaned[labeled_sample_cleaned["sqft_context"].notna()][["sqft_context"]]

,sqft_context
258,"tral Coast living in this stunning 4,146 squar..."
301,"and expansive sunset horizons. The 3,869 squar..."
944,"nal versatility. Inside, more than 1,500 squar..."
288,"ring 5 bedrooms, 4.5 bathrooms and 3,606 squar..."
229,"nt single-family home spans nearly 3,300 squar..."
...,...
123,"lit air conditioner, approximately 1400 square..."
528,"s 3 beds and 3 baths in a generous 2,201 squar..."
628,"ersatile home offers approximately 1,940 squar..."
19,", this 3-story contemporary offers 1,655 squar..."


In [540]:
def sqft_phrase(text):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    return match.group(0) if match else None

In [541]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)

In [542]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(87)

In [543]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)
labeled_sample_cleaned["sqft"] = None

In [544]:
df_cleaned['cleaned_remarks'].loc[258]

'Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible

In [545]:
labeled_sample_cleaned.head(10)

,beds,baths,price,cleaned_remarks,sqft,sqft_context,sqft_phrase
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits...",None,NaN,NaN
255,2.0,3.0,389000,Multi-level townhouse looking for a little lov...,None,NaN,NaN
258,4.0,4.0,2625000,Experience refined Central Coast living in thi...,None,"tral Coast living in this stunning 4,146 squar...","4,146 square feet"
301,3.0,4.0,3325000,Wine country estate set across 52 acres in the...,None,"and expansive sunset horizons. The 3,869 squar...","3,869 square feet"
233,3.0,2.0,650000,Welcome to this beautifully upgraded Connect f...,None,NaN,NaN
143,3.0,3.0,1460600,"Welcome to this spectacular 3-bedroom, 3-bath ...",None,NaN,NaN
378,4.0,3.0,950000,Welcome to 70900 Jasmine Lane located in beaut...,None,NaN,NaN
679,3.0,2.0,429000,"Welcome to this stunning, move-in-ready reside...",None,NaN,NaN
944,4.0,2.0,920000,"Welcome to 7039 Sylvia Avenue, a beautifully u...",None,"nal versatility. Inside, more than 1,500 squar...","1,500 square feet"
37,3.0,2.0,599900,GRAND TERRACE SINGLE-STORY-Welcome home to thi...,None,NaN,NaN


In [546]:
def label_sqft(phrase):
    if pd.isna(phrase):
        return None

    match = re.search(r'(\d[\d,]*)', phrase)

    if match:
        return int(match.group(1).replace(",", ""))

    return None

In [547]:
labeled_sample_cleaned["sqft"] = labeled_sample_cleaned["sqft_phrase"].apply(label_sqft)

In [548]:
labeled_sample_cleaned['sqft'].notna().sum()

np.int64(87)

In [549]:
labeled_sample_cleaned.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'sqft_context',
       'sqft_phrase'],
      dtype='str')

In [550]:
labeled_sample_cleaned['sqft_context'].notna().sum()

np.int64(87)

In [551]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(87)

In [552]:
labeled_cleaned = labeled_sample_cleaned.drop(columns=['sqft_phrase', 'sqft_context'])

In [553]:
labeled_cleaned["sqft"] = labeled_cleaned["sqft"].fillna("None")

In [554]:
labeled_cleaned['sqft'].isna().sum()

np.int64(0)

Amendities (ground truth)

In [555]:
# Using taxonomy as a helper/resource
def suggest_amenities(text):
    suggestions = []

    for term in taxonomy_terms:
        if term.lower() in text.lower():
            suggestions.append(term)

    return suggestions

taxonomy_terms = sorted(
    [term["term"] for term in taxonomy["terms"]],
    key=len,
    reverse=True
)

In [556]:
labeled_cleaned["candidate_amenities"] = (
    labeled_cleaned["cleaned_remarks"]
    .apply(suggest_amenities)
)

In [557]:
labeled_cleaned['candidate_amenities']

606    [flooring throughout, sliding glass doors, mou...
255           [primary bedroom, dining room, car garage]
258    [spacious primary suite, primary suite offers,...
301    [primary suite serves, quartz countertops, pri...
233    [stainless steel appliances, conveniently loca...
                             ...                        
19     [primary suite offers, quartz countertops, pan...
449    [effortless entertaining, fitness center, prim...
755    [flooring throughout, quartz countertops, rece...
148    [laminate flooring, covered patio, living room...
997    [primary bedroom, full bathrooms, indoor laund...
Name: candidate_amenities, Length: 250, dtype: object

Go through the suggested amenities

In [558]:
#labeled_cleaned.to_csv("../data/labeled_cleaned.csv")

Keep in mind: The amentities are meant to represent the features that the property offers, not surrounding words or phrases that happen to contain those features. I will also avoid any marketing phrases. An amenity will represent a tangible property feature, space, service, or location advantage that the property provides. So, if there are any phrases that are not supportive of this, they will be removed (to keep ground truth)

In [559]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 86),
 ('primary suite', 72),
 ('car garage', 67),
 ('natural light', 63),
 ('floor plan', 63),
 ('living room', 52),
 ('everyday living', 44),
 ('conveniently located', 41),
 ('located near', 39),
 ('full bath', 38),
 ('recessed lighting', 35),
 ('easy access', 33),
 ('stainless steel', 30),
 ('living spaces', 30),
 ('conveniently located near', 28),
 ('home office', 27),
 ('spacious living', 27),
 ('full bathroom', 27),
 ('family room', 25),
 ('quartz countertops', 25),
 ('stainless steel appliances', 25),
 ('outdoor space', 24),
 ('suite offers', 24),
 ('bath home', 24),
 ('dining room', 23),
 ('fully remodeled', 23),
 ('laundry room', 22),
 ('primary suite offers', 21),
 ('covered patio', 20),
 ('granite countertops', 20),
 ('updated kitchen', 20),
 ('open floor', 20),
 ('ideally located', 20),
 ('abundant natural light', 19),
 ('vaulted ceilings', 19),
 ('direct access', 19),
 ('mountain views', 18),
 ('primary bedroom', 18),
 ('open floor plan', 18),
 ('golf cour

In [560]:
len(amenity_counts)

153

In [561]:
removed_amenities= [
 'everyday living', 'conveniently located', 'located near', 
 'easy access', 'conveniently located near', 'ideally located', 
 'spacious primary', 'flooring throughout', 'fully remodeled', 
 'primary suite offers', 'suite offers', 'direct access', 'flows seamlessly', 
 'convenient access', 'everyday convenience', 'located within', 
 'prime location', 'brand new', 'ample space', 'home located', 
 'beautifully remodeled', 'ideally located near', 'comfortable living space', 
 'enjoy access', 'endless possibilities', 'seamless flow', 'effortless entertaining', 
 'primary suite serves', 'suite serves', 'kitchen showcases', 'everyday comfort', 
 'every detail', 'primary suite features', 'suite features','suite includes', 
 'floor plan features', 'peaceful retreat', 'kitchen featuring', 'level features', 
 'spacious home', 'bright open', 'patio perfect', 'kitchen equipped',
 'features spacious','bedrooms including', 'seamless living', 'located minutes',
 'feet living space', 'space home'
]

In [562]:
labeled_cleaned["candidate_amenities"] = labeled_cleaned["candidate_amenities"].apply(
    lambda amenities: [
        x for x in amenities
        if x not in removed_amenities
    ]
)

In [563]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 86),
 ('primary suite', 72),
 ('car garage', 67),
 ('natural light', 63),
 ('floor plan', 63),
 ('living room', 52),
 ('full bath', 38),
 ('recessed lighting', 35),
 ('stainless steel', 30),
 ('living spaces', 30),
 ('home office', 27),
 ('spacious living', 27),
 ('full bathroom', 27),
 ('family room', 25),
 ('quartz countertops', 25),
 ('stainless steel appliances', 25),
 ('outdoor space', 24),
 ('bath home', 24),
 ('dining room', 23),
 ('laundry room', 22),
 ('covered patio', 20),
 ('granite countertops', 20),
 ('updated kitchen', 20),
 ('open floor', 20),
 ('abundant natural light', 19),
 ('vaulted ceilings', 19),
 ('mountain views', 18),
 ('primary bedroom', 18),
 ('open floor plan', 18),
 ('golf course', 18),
 ('gated community', 18),
 ('plank flooring', 17),
 ('freeway access', 17),
 ('fitness center', 16),
 ('wood flooring', 15),
 ('private balcony', 15),
 ('main level', 15),
 ('gourmet kitchen', 14),
 ('sparkling pool', 14),
 ('parking space', 14),
 ('spacious

In [564]:
len(amenity_counts)

108

In [567]:
labeled_cleaned = labeled_cleaned.rename(columns={'candidate_amenities': 'amenities'})

In [568]:
labeled_cleaned.head()

,beds,baths,price,cleaned_remarks,sqft,amenities
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits...",None,"[sliding glass doors, mountain views, second b..."
255,2.0,3.0,389000,Multi-level townhouse looking for a little lov...,None,"[primary bedroom, dining room, car garage]"
258,4.0,4.0,2625000,Experience refined Central Coast living in thi...,4146.0,"[spacious primary suite, granite countertops, ..."
301,3.0,4.0,3325000,Wine country estate set across 52 acres in the...,3869.0,"[quartz countertops, primary suite]"
233,3.0,2.0,650000,Welcome to this beautifully upgraded Connect f...,None,"[stainless steel appliances, quartz countertop..."


Using the extractor class

In [569]:
labeled_extract = labeled_cleaned.copy()

In [570]:
labeled_extract = labeled_extract.drop(columns=['beds', 'baths', 'price', 'sqft', 'amenities'])

In [572]:
labeled_extract.head()

,cleaned_remarks
606,"This inviting 2-bedroom, 1-bath residence sits..."
255,Multi-level townhouse looking for a little lov...
258,Experience refined Central Coast living in thi...
301,Wine country estate set across 52 acres in the...
233,Welcome to this beautifully upgraded Connect f...


In [573]:
labeled_extract.shape

(250, 1)

In [578]:
labeled_extract['extracted_bd'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bedrooms
)

In [579]:
labeled_extract['extracted_bd'].isna().sum()

np.int64(166)

In [581]:
labeled_extract['extracted_br'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bathrooms
)

ValueError: invalid literal for int() with base 10: '4.5'

In [582]:
labeled_extract['extracted_price'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_price
)

In [586]:
labeled_extract['extracted_price'].isna().sum()

np.int64(234)

In [583]:
labeled_extract['extracted_sqft'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_sqft
)

ValueError: invalid literal for int() with base 10: 'square feet'

In [584]:
labeled_extract['extracted_amenities'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_amenities
)

In [585]:
labeled_extract['extracted_amenities'].isna().sum()

np.int64(0)